<a href="https://colab.research.google.com/github/Conscht/MNIST_Curation_Repo/blob/main/MNIST_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install all dependencies and import libraries.

In [1]:
!uv pip install fiftyone==1.7.0 torch==2.6.0 torchvision==0.21 numpy==2.0.2 open-clip-torch==3.2.0

Using Python 3.12.12 environment at: /usr
Resolved 145 packages in 2.30s
Prepared 65 packages in 1m 07s
Uninstalled 18 packages in 822ms
Installed 65 packages in 510ms
 + argcomplete==3.6.3
 + async-lru==2.0.5
 + bcrypt==5.0.0
 + boto3==1.40.64
 + botocore==1.40.64
 + choreographer==1.2.0
 + dacite==1.7.0
 + deprecated==1.3.1
 + dnspython==2.8.0
 + fiftyone==1.7.0
 + fiftyone-brain==0.21.3
 + fiftyone-db==1.3.0
 + ftfy==6.3.1
 + graphql-core==3.2.6
 + hypercorn==0.17.3
 + inflate64==1.0.3
 + jmespath==1.0.1
 + jsonlines==4.0.0
 + kaleido==1.1.0
 + lia-web==0.2.3
 + logistro==2.0.1
 + mongoengine==0.29.1
 + motor==3.6.1
 + multivolumefile==0.2.3
 - nvidia-cublas-cu12==12.6.4.1
 + nvidia-cublas-cu12==12.4.5.8
 - nvidia-cuda-cupti-cu12==12.6.80
 + nvidia-cuda-cupti-cu12==12.4.127
 - nvidia-cuda-nvrtc-cu12==12.6.77
 + nvidia-cuda-nvrtc-cu12==12.4.127
 - nvidia-cuda-runtime-cu12==12.6.77
 + nvidia-cuda-runtime-cu12==12.4.127
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.1.0.70
 - 

In [2]:
import fiftyone
import torch
import torchvision
import numpy
import open_clip
(fiftyone.__version__, torch.__version__,
 torchvision.__version__, numpy.__version__,
 open_clip.__version__)

/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


('1.7.0', '2.6.0+cu124', '0.21.0+cu124', '2.0.2', '3.2.0')


**Task 1: Create visualizations of the embedding space with PCA and UMAP in FiftyOne**


For this we will need a baseline model that gives its embedding space. We will use the CLIP model


Install fityone plugins.

In [4]:
!fiftyone plugins download \
    https://github.com/voxel51/fiftyone-plugins \
    --plugin-names @voxel51/evaluation


Skipping existing plugin '@voxel51/evaluation'

Copying plugin '@jacobmarks/albumentations_augmentation' to '/root/fiftyone/__plugins__/@jacobmarks/albumentations_augmentation'


In [ ]:
!fiftyone plugins download https://github.com/jacobmarks/fiftyone-albumentations-plugin

*Imports*

In [5]:
import os

# Set environment variables for reproducibility BEFORE importing torch
os.environ['PYTHONHASHSEED'] = '51'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fun
import torchvision.transforms.v2 as transforms
import fiftyone as fo
import fiftyone.zoo as foz
import fiftyone.brain as fob
from torch.utils.data import Dataset, ConcatDataset
from fiftyone import ViewField as F
import fiftyone.utils.random as four
from tqdm import tqdm
from torch.optim import Adam
from pathlib import Path
import matplotlib.pyplot as plt
import gc
import albumentations as A
import cv2
import random
from typing import Optional, Dict, Tuple, Any

**Load ad visualize data using FortyOne**

In [6]:
test_dataset = foz.load_zoo_dataset("mnist", split='test')

INFO:fiftyone.zoo.datasets:Downloading split 'test' to '/root/fiftyone/mnist/test'
100%|██████████| 9.91M/9.91M [00:00<00:00, 19.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 605kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 5.62MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.68MB/s]

   2% |/------------|   184/10000 [114.5ms elapsed, 6.1s remaining, 1.6K samples/s] 

 100% |█████████████| 10000/10000 [4.1s elapsed, 0s remaining, 2.5K samples/s]      


INFO:eta.core.utils: 100% |█████████████| 10000/10000 [4.1s elapsed, 0s remaining, 2.5K samples/s]      


Dataset info written to '/root/fiftyone/mnist/info.json'


...

INFO:fiftyone.zoo.datasets:Loading 'mnist' split 'test'


 100% |█████████████| 10000/10000 [9.0s elapsed, 0s remaining, 1.3K samples/s]        


INFO:eta.core.utils: 100% |█████████████| 10000/10000 [9.0s elapsed, 0s remaining, 1.3K samples/s]        


Dataset 'mnist-test' created


INFO:fiftyone.zoo.datasets:Dataset 'mnist-test' created


Launch the App

In [7]:
session = fo.launch_app(test_dataset, auto=False)


Could not connect session, trying again in 10 seconds

Session launched. Run `session.show()` to open the App in a cell output.


INFO:fiftyone.core.session.session:Session launched. Run `session.show()` to open the App in a cell output.



Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.7.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



INFO:fiftyone.core.session.session:
Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.7.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



In [8]:
test_dataset.compute_metadata()

Computing metadata...


INFO:fiftyone.core.metadata:Computing metadata...


 100% |█████████████| 10000/10000 [4.9s elapsed, 0s remaining, 2.4K samples/s]      


INFO:eta.core.utils: 100% |█████████████| 10000/10000 [4.9s elapsed, 0s remaining, 2.4K samples/s]      


In [11]:
session.refresh()
print(session.url)

https://5151-m-s-1ujj2sy0ar1fn-b.us-west3-0.prod.colab.dev?polling=true


We could use a simple modle or train one from scratch, but to really find critical samples, it is necessary that our model has already great classificaiton abilities. Limiting the false detected samples to actually faulty samples.

For this, we use Open AIs CLIP model.


1.   the model is obtained through foz.load_zoo_model(""clip-vit-base32-torch"")
2.   We get the embeddings via the compute_embeddings()




In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = foz.load_zoo_model("clip-vit-base32-torch",
                                device=device)
print(f"The model is loaded on {clip_model._device}")

INFO:fiftyone.core.models:Downloading model from 'https://openaipublic.azureedge.net/clip/models/40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af/ViT-B-32.pt'...


 100% |██████|    2.6Gb/2.6Gb [9.9s elapsed, 0s remaining, 222.1Mb/s]       


INFO:eta.core.utils: 100% |██████|    2.6Gb/2.6Gb [9.9s elapsed, 0s remaining, 222.1Mb/s]       
/usr/local/lib/python3.12/dist-packages/fiftyone/utils/clip/tokenizer.py:107: SyntaxWarning: invalid escape sequence '\p'
  + """[\p{L}]+|[\p{N}]|[^\s\p{L}\p{N}]+""",


INFO:fiftyone.utils.clip.zoo:Downloading CLIP tokenizer...


 100% |█████|   10.4Mb/10.4Mb [104.0ms elapsed, 0s remaining, 99.6Mb/s]     


INFO:eta.core.utils: 100% |█████|   10.4Mb/10.4Mb [104.0ms elapsed, 0s remaining, 99.6Mb/s]     


The model is loaded on cpu


Compute all emebddings of the dataset

In [ ]:
clip_embeddings = test_dataset.compute_embeddings(model=clip_model,
                                        batch_size=512,
                                        num_workers=2)

  46% |█████|-------|  4608/10000 [18.1m elapsed, 21.2m remaining, 4.2 samples/s] 

Link samples and embeddings

In [ ]:
for index, sample in enumerate(test_dataset):
    sample["clip_embeddings"] = clip_embeddings[index]
    sample.save()

Use PCA and UMAP to visualize

In [ ]:
pca_visualization = fob.compute_visualization(test_dataset,
                                              method="pca",
                                              embeddings="clip_embeddings",
                                              num_dims=2,
                                              brain_key="pca_visualization_clip_embeds")

In [ ]:
umap_visualization = fob.compute_visualization(test_dataset,
                                              method="umap",
                                              embeddings="clip_embeddings",
                                              num_dims=2,
                                              brain_key="umap_visualization_clip_embeds")

In [ ]:
session.refresh()
print(session.url)

**Task 2: Curate the MNIST dataset to find 'questionable' digits (e.g. the Spaghetti Nine) as a FiftyOne dataset**



Create a second classifier that includes the IDK ('I Don't Know') label for digits of questionable quality

Options: Binary classifier or multi-label classification including the IDK class (10 MNIST classes + 1 IDK class)



Publish your curated version of MNIST on HuggingFace and the notebook you used to curate it as a public GitHub repository



The notebook here provides guidance on most tasks. Notice that the exploration of questionable and low quality examples on it is not exhaustive.